# RepairScope-X — Notebook 03 FINAL
## RepairBench-X: Controlled Failures + Exact Verifier Audit

**Fully standalone.** No ZIP, previous notebook, model checkpoint, `src/`, or GPU is required.

This notebook intentionally separates **benchmark construction** from the custom-model runtime that caused the earlier compatibility failures.

### Pilot datasets
- **GSM8K** — `openai/gsm8k`, `main`.
- **SVAMP** — `ChilleD/SVAMP`, chosen because this release exposes `Body`, `Question`, `Equation`, and `Answer`.
- **Fresh Countdown** — generated locally from frozen seeds and verified by a safe arithmetic evaluator.

For each public dataset the notebook first asks the Hugging Face Hub for the current repository SHA, records it, and then loads that exact revision.

### Controlled failure classes
- **F1:** local wrong final numeric answer.
- **F2:** coupled arithmetic-expression + final-answer corruption.
- **F3:** final-answer marker/format failure.
- **F4:** global truncation removing the verifiable answer.

A controlled failure is retained only when:
1. its source response passes the exact verifier; and
2. its corrupted response fails that verifier.

### Why MATH-500 is deferred here
MATH-500 mixes numeric, symbolic, algebraic, geometric, and LaTeX answers. This notebook refuses to silently score it with a fragile "last number" parser. A later notebook should add MATH-500 only after symbolic equivalence is independently validated.

In [ ]:
# Stable data dependencies only. No Transformers/custom-model stack.
!pip -q install -U "datasets>=3.3,<5" "huggingface_hub>=0.30" pandas pyarrow

## 1. Environment and deterministic output location

In [ ]:
import os, sys, json, math, re, ast, random, hashlib, platform, difflib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

SEED = 1703
random.seed(SEED)
np.random.seed(SEED)

SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/RepairScopeX/Notebook03")
else:
    OUT_DIR = Path("/content/RepairScopeX_Notebook03")

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("Seed:", SEED)
print("Output:", OUT_DIR)

## 2. Strict primary numeric verifier

Later model prompts will be required to end with:

`FINAL_ANSWER: <number>`

The verifier reads that field only. Numbers elsewhere in the reasoning are ignored.

In [ ]:
FINAL_RE = re.compile(
    r"(?im)^[ \t]*FINAL_ANSWER[ \t]*:[ \t]*"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)"
    r"[ \t]*$"
)

def canonical_number(x):
    x = float(x)
    if not math.isfinite(x):
        raise ValueError("Non-finite target")
    if x == 0:
        x = 0.0
    return str(int(x)) if x.is_integer() else format(x, ".12g")

def extract_strict_final_answer(text):
    matches = FINAL_RE.findall(str(text))
    if not matches:
        return None
    try:
        value = float(matches[-1])
        return value if math.isfinite(value) else None
    except Exception:
        return None

def verify_numeric_response(text, target, atol=1e-8, rtol=1e-8):
    pred = extract_strict_final_answer(text)
    target = float(target)
    passed = (
        pred is not None
        and math.isfinite(target)
        and math.isclose(pred, target, abs_tol=atol, rel_tol=rtol)
    )
    return {"passed": bool(passed), "prediction": pred, "target": target}

assert verify_numeric_response("2+2=4\nFINAL_ANSWER: 4", 4)["passed"]
assert not verify_numeric_response("2+2=4\nANSWER: 4", 4)["passed"]
assert not verify_numeric_response("reasoning contains 4 only", 4)["passed"]
assert verify_numeric_response("FINAL_ANSWER: -2.5", -2.5)["passed"]

print("Strict verifier unit tests: PASS")

## 3. Safe arithmetic evaluator

Controlled expression mutations are evaluated through a small Python AST allow-list.
No arbitrary dataset/model text is passed to `eval`.

In [ ]:
BIN_OPS = {
    ast.Add: lambda a, b: a + b,
    ast.Sub: lambda a, b: a - b,
    ast.Mult: lambda a, b: a * b,
    ast.Div: lambda a, b: a / b,
}
UNARY_OPS = {
    ast.UAdd: lambda a: +a,
    ast.USub: lambda a: -a,
}

def safe_arithmetic(expr):
    tree = ast.parse(str(expr).strip(), mode="eval")

    def visit(node):
        if isinstance(node, ast.Expression):
            return visit(node.body)
        if isinstance(node, ast.Constant):
            if isinstance(node.value, (int, float)):
                value = float(node.value)
                if not math.isfinite(value):
                    raise ValueError("non-finite constant")
                return value
            raise ValueError("non-numeric constant")
        if isinstance(node, ast.UnaryOp) and type(node.op) in UNARY_OPS:
            return UNARY_OPS[type(node.op)](visit(node.operand))
        if isinstance(node, ast.BinOp) and type(node.op) in BIN_OPS:
            value = BIN_OPS[type(node.op)](visit(node.left), visit(node.right))
            if not math.isfinite(value):
                raise ValueError("non-finite result")
            return value
        raise ValueError(f"Disallowed node: {type(node).__name__}")

    return visit(tree)

assert safe_arithmetic("(12 + 3) * 2") == 30
assert safe_arithmetic("10 / 4") == 2.5
try:
    safe_arithmetic("__import__('os').system('echo unsafe')")
    raise AssertionError("Unsafe expression was not rejected")
except ValueError:
    pass

print("Safe arithmetic tests: PASS")

## 4. Freeze current public dataset revisions, then load them

In [ ]:
from datasets import load_dataset
from huggingface_hub import HfApi

api = HfApi()

DATASET_IDS = {
    "gsm8k": "openai/gsm8k",
    "svamp": "ChilleD/SVAMP",
}

DATASET_SHAS = {}
for name, repo_id in DATASET_IDS.items():
    info = api.dataset_info(repo_id)
    DATASET_SHAS[name] = info.sha
    print(name, "->", repo_id, "@", info.sha)

gsm_train = load_dataset(
    DATASET_IDS["gsm8k"], "main", split="train",
    revision=DATASET_SHAS["gsm8k"]
)
gsm_test = load_dataset(
    DATASET_IDS["gsm8k"], "main", split="test",
    revision=DATASET_SHAS["gsm8k"]
)

svamp_train = load_dataset(
    DATASET_IDS["svamp"], split="train",
    revision=DATASET_SHAS["svamp"]
)
svamp_test = load_dataset(
    DATASET_IDS["svamp"], split="test",
    revision=DATASET_SHAS["svamp"]
)

print("GSM8K:", len(gsm_train), len(gsm_test), gsm_train.column_names)
print("SVAMP:", len(svamp_train), len(svamp_test), svamp_train.column_names)

assert {"question", "answer"}.issubset(gsm_train.column_names)
assert {"Body", "Question", "Equation", "Answer"}.issubset(svamp_train.column_names)

print("Dataset schema checks: PASS")

## 5. Dataset-specific ground-truth parsers

In [ ]:
SIMPLE_NUM_RE = re.compile(
    r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?"
)

def parse_gsm8k_target(answer):
    if "####" not in answer:
        raise ValueError("GSM8K answer lacks ####")
    raw = answer.rsplit("####", 1)[1].strip().replace(",", "")
    if not re.fullmatch(
        r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?", raw
    ):
        raise ValueError(f"Non-numeric GSM8K target: {raw!r}")
    value = float(raw)
    if not math.isfinite(value):
        raise ValueError("non-finite")
    return value

def parse_svamp_target(answer):
    nums = SIMPLE_NUM_RE.findall(str(answer).strip().replace(",", ""))
    if not nums:
        raise ValueError(f"No numeric SVAMP target: {answer!r}")
    value = float(nums[-1])
    if not math.isfinite(value):
        raise ValueError("non-finite")
    return value

for ex in gsm_train.select(range(10)):
    parse_gsm8k_target(ex["answer"])

for ex in svamp_train.select(range(10)):
    parse_svamp_target(ex["Answer"])

print("Ground-truth parser checks: PASS")

## 6. Frozen subset and split policy

- Router development/calibration comes only from original **training** splits.
- Confirmatory examples come only from original **test** splits.
- Countdown uses disjoint seed namespaces for train/calibration/test.

In [ ]:
GSM_TRAIN_N, GSM_CAL_N, GSM_TEST_N = 96, 32, 64
SVAMP_TRAIN_N, SVAMP_CAL_N, SVAMP_TEST_N = 96, 32, 64
COUNTDOWN_TRAIN_N, COUNTDOWN_CAL_N, COUNTDOWN_TEST_N = 128, 48, 64

def deterministic_indices(n_total, n_pick, seed):
    if n_pick > n_total:
        raise ValueError((n_pick, n_total))
    rng = np.random.default_rng(seed)
    return sorted(rng.choice(n_total, size=n_pick, replace=False).tolist())

gsm_dev_idx = deterministic_indices(
    len(gsm_train), GSM_TRAIN_N + GSM_CAL_N, SEED + 11
)
gsm_test_idx = deterministic_indices(
    len(gsm_test), GSM_TEST_N, SEED + 12
)
svamp_dev_idx = deterministic_indices(
    len(svamp_train), SVAMP_TRAIN_N + SVAMP_CAL_N, SEED + 21
)
svamp_test_idx = deterministic_indices(
    len(svamp_test), SVAMP_TEST_N, SEED + 22
)

print("Frozen subset selection: PASS")

## 7. Fresh Countdown generation

In [ ]:
def generate_countdown(n, seed, partition):
    rng = random.Random(seed)
    rows = []

    for i in range(n):
        a, b, c = [rng.randint(2, 30) for _ in range(3)]
        op1 = rng.choice(["+", "-", "*"])
        op2 = rng.choice(["+", "-", "*"])

        expr = f"({a} {op1} {b}) {op2} {c}"
        target = safe_arithmetic(expr)

        rows.append({
            "dataset": "countdown_fresh",
            "source_split": "generated",
            "partition": partition,
            "source_index": i,
            "problem_id": f"countdown:{partition}:{seed}:{i}",
            "prompt": (
                f"Use the numbers {a}, {b}, and {c} exactly once to produce "
                f"the target {canonical_number(target)}. Return a valid "
                "arithmetic expression and end with `FINAL_ANSWER: <number>`."
            ),
            "target": target,
            "reference_reasoning": f"Expression: {expr}",
            "reference_expression": expr,
            "metadata": {
                "numbers": [a, b, c],
                "generator_seed": seed,
            },
        })

    return rows

countdown_train = generate_countdown(
    COUNTDOWN_TRAIN_N, SEED + 101, "train"
)
countdown_cal = generate_countdown(
    COUNTDOWN_CAL_N, SEED + 102, "calibration"
)
countdown_test = generate_countdown(
    COUNTDOWN_TEST_N, SEED + 103, "test"
)

assert set(x["problem_id"] for x in countdown_train).isdisjoint(
    set(x["problem_id"] for x in countdown_test)
)

print("Fresh Countdown: PASS")

## 8. Normalize all sources into one schema

In [ ]:
def make_gsm_record(ex, source_index, source_split, partition):
    target = parse_gsm8k_target(ex["answer"])
    reasoning = ex["answer"].rsplit("####", 1)[0].strip()

    return {
        "dataset": "gsm8k",
        "source_split": source_split,
        "partition": partition,
        "source_index": int(source_index),
        "problem_id": f"gsm8k:{source_split}:{source_index}",
        "prompt": ex["question"],
        "target": target,
        "reference_reasoning": reasoning,
        "reference_expression": None,
        "metadata": {},
    }

def make_svamp_record(ex, source_index, source_split, partition):
    target = parse_svamp_target(ex["Answer"])
    body = str(ex["Body"]).strip()
    question = str(ex["Question"]).strip()
    equation = str(ex["Equation"]).strip()

    return {
        "dataset": "svamp",
        "source_split": source_split,
        "partition": partition,
        "source_index": int(source_index),
        "problem_id": f"svamp:{source_split}:{ex.get('ID', source_index)}",
        "prompt": (body + " " + question).strip(),
        "target": target,
        "reference_reasoning": f"Equation: {equation}",
        "reference_expression": equation,
        "metadata": {
            "svamp_id": ex.get("ID"),
            "svamp_type": ex.get("Type"),
        },
    }

sources = []

for rank, idx in enumerate(gsm_dev_idx):
    partition = "train" if rank < GSM_TRAIN_N else "calibration"
    sources.append(make_gsm_record(gsm_train[idx], idx, "train", partition))

for idx in gsm_test_idx:
    sources.append(make_gsm_record(gsm_test[idx], idx, "test", "test"))

for rank, idx in enumerate(svamp_dev_idx):
    partition = "train" if rank < SVAMP_TRAIN_N else "calibration"
    sources.append(make_svamp_record(svamp_train[idx], idx, "train", partition))

for idx in svamp_test_idx:
    sources.append(make_svamp_record(svamp_test[idx], idx, "test", "test"))

sources.extend(countdown_train)
sources.extend(countdown_cal)
sources.extend(countdown_test)

print("Normalized source problems:", len(sources))

## 9. Build verified-correct canonical responses

In [ ]:
def canonical_correct_response(record):
    reasoning = str(record["reference_reasoning"]).strip()
    final = f"FINAL_ANSWER: {canonical_number(record['target'])}"
    return f"{reasoning}\n{final}".strip()

for record in sources:
    response = canonical_correct_response(record)
    vr = verify_numeric_response(response, record["target"])
    if not vr["passed"]:
        raise AssertionError(
            f"Reference failed verifier: {record['problem_id']} {vr}"
        )
    record["canonical_correct_response"] = response

print("Verified-correct source responses:", len(sources))

## 10. Exact changed-span recording

In [ ]:
def changed_spans(original, corrupted):
    matcher = difflib.SequenceMatcher(a=original, b=corrupted)
    spans = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag != "equal":
            spans.append({
                "tag": tag,
                "original_start": int(i1),
                "original_end": int(i2),
                "corrupted_start": int(j1),
                "corrupted_end": int(j2),
                "original_text": original[i1:i2],
                "corrupted_text": corrupted[j1:j2],
            })

    return spans

assert changed_spans("abc", "adc")
print("Changed-span helper: PASS")

## 11. Binary arithmetic-expression mutation

In [ ]:
def binary_operator_positions(expr):
    positions = []

    for i, ch in enumerate(expr):
        if ch not in "+-*/":
            continue

        left = i - 1
        while left >= 0 and expr[left].isspace():
            left -= 1

        right = i + 1
        while right < len(expr) and expr[right].isspace():
            right += 1

        if left < 0 or right >= len(expr):
            continue

        if ch in "+-" and expr[left] in "(+-*/":
            continue

        if (expr[left].isdigit() or expr[left] == ")") and (
            expr[right].isdigit() or expr[right] == "("
        ):
            positions.append(i)

    return positions

def mutate_expression(expr, correct_target):
    expr = str(expr).strip()

    alternatives = {
        "+": ["-", "*", "/"],
        "-": ["+", "*", "/"],
        "*": ["+", "-", "/"],
        "/": ["*", "+", "-"],
    }

    for pos in binary_operator_positions(expr):
        old = expr[pos]

        for new_op in alternatives[old]:
            candidate = expr[:pos] + new_op + expr[pos + 1:]

            try:
                value = safe_arithmetic(candidate)
            except Exception:
                continue

            if not math.isfinite(value):
                continue

            if math.isclose(
                value, float(correct_target), abs_tol=1e-8, rel_tol=1e-8
            ):
                continue

            return {
                "original_expression": expr,
                "mutated_expression": candidate,
                "mutated_value": value,
                "operator_position": pos,
                "old_operator": old,
                "new_operator": new_op,
            }

    return None

demo = mutate_expression("(12 + 3) * 2", 30)
assert demo is not None
assert not math.isclose(demo["mutated_value"], 30)

print("Expression mutation helper: PASS")

## 12. Controlled failure generators

F2 uses:
- the explicit SVAMP/Countdown expression when available;
- the final GSM8K `<<EXPR=RESULT>>` calculator annotation otherwise.

In [ ]:
GSM_CALC_RE = re.compile(r"<<([^<>]+)=([^<>]+)>>")

def failure_f1_final_numeric(record):
    original = record["canonical_correct_response"]
    wrong = float(record["target"]) + 1.0

    old = f"FINAL_ANSWER: {canonical_number(record['target'])}"
    new = f"FINAL_ANSWER: {canonical_number(wrong)}"

    if old not in original:
        return None, "final_line_not_found"

    corrupted = original.rsplit(old, 1)[0] + new

    return {
        "failure_kind": "F1_final_numeric",
        "topology": "single_local",
        "corrupted_response": corrupted,
        "metadata": {"wrong_target": wrong},
    }, None

def failure_f3_marker(record):
    original = record["canonical_correct_response"]

    if "FINAL_ANSWER:" not in original:
        return None, "marker_not_found"

    prefix, suffix = original.rsplit("FINAL_ANSWER:", 1)
    corrupted = prefix + "ANSWER:" + suffix

    return {
        "failure_kind": "F3_verifier_marker",
        "topology": "single_local_format",
        "corrupted_response": corrupted,
        "metadata": {},
    }, None

def failure_f4_truncation(record):
    original = record["canonical_correct_response"]
    lines = original.splitlines()

    if len(lines) < 2:
        return None, "too_few_lines"

    keep = max(1, len(lines) // 2)
    corrupted = "\n".join(lines[:keep]).strip()

    if not corrupted:
        return None, "empty_after_truncation"

    return {
        "failure_kind": "F4_global_truncation",
        "topology": "global",
        "corrupted_response": corrupted,
        "metadata": {
            "kept_lines": keep,
            "original_lines": len(lines),
        },
    }, None

def failure_f2_coupled(record):
    original = record["canonical_correct_response"]
    target = float(record["target"])
    expr = record.get("reference_expression")

    if not expr and record["dataset"] == "gsm8k":
        annotations = list(
            GSM_CALC_RE.finditer(record["reference_reasoning"])
        )
        if annotations:
            expr = annotations[-1].group(1).strip()

    if not expr:
        return None, "no_mutable_expression"

    mutation = mutate_expression(expr, target)

    if mutation is None:
        return None, "no_valid_operator_mutation"

    wrong = mutation["mutated_value"]
    corrupted = original

    if expr not in corrupted:
        return None, "expression_not_found_in_response"

    corrupted = corrupted.replace(
        expr, mutation["mutated_expression"], 1
    )

    # If the final GSM8K calculator annotation's result equals the true target,
    # make that annotation consistent with the mutated expression too.
    if record["dataset"] == "gsm8k":
        matches = list(GSM_CALC_RE.finditer(corrupted))
        if matches:
            m = matches[-1]
            try:
                rhs_num = float(m.group(2).strip().replace(",", ""))
            except Exception:
                rhs_num = None

            if rhs_num is not None and math.isclose(
                rhs_num, target, abs_tol=1e-8, rel_tol=1e-8
            ):
                new_annotation = (
                    f"<<{m.group(1)}={canonical_number(wrong)}>>"
                )
                corrupted = (
                    corrupted[:m.start()]
                    + new_annotation
                    + corrupted[m.end():]
                )

    old_final = f"FINAL_ANSWER: {canonical_number(target)}"
    new_final = f"FINAL_ANSWER: {canonical_number(wrong)}"

    if old_final not in corrupted:
        return None, "final_line_not_found_after_mutation"

    corrupted = corrupted.rsplit(old_final, 1)[0] + new_final

    return {
        "failure_kind": "F2_coupled_expression_final",
        "topology": "two_region_coupled",
        "corrupted_response": corrupted,
        "metadata": mutation,
    }, None

print("Failure generators: READY")

## 13. Generate RepairBench-X controlled failures

Every candidate is verified after corruption. A candidate that still passes is discarded.

In [ ]:
FAILURE_BUILDERS = [
    failure_f1_final_numeric,
    failure_f2_coupled,
    failure_f3_marker,
    failure_f4_truncation,
]

failures = []
discarded = []

for record in sources:
    original = record["canonical_correct_response"]
    source_vr = verify_numeric_response(original, record["target"])
    assert source_vr["passed"]

    for builder in FAILURE_BUILDERS:
        candidate, reason = builder(record)

        if candidate is None:
            discarded.append({
                "problem_id": record["problem_id"],
                "builder": builder.__name__,
                "reason": reason,
            })
            continue

        corrupted = candidate["corrupted_response"]
        corrupted_vr = verify_numeric_response(
            corrupted, record["target"]
        )

        if corrupted_vr["passed"]:
            discarded.append({
                "problem_id": record["problem_id"],
                "builder": builder.__name__,
                "reason": "corruption_still_passed_primary_verifier",
            })
            continue

        spans = changed_spans(original, corrupted)

        if not spans:
            discarded.append({
                "problem_id": record["problem_id"],
                "builder": builder.__name__,
                "reason": "no_changed_span",
            })
            continue

        failure_id = (
            f"{record['problem_id']}::{candidate['failure_kind']}"
        )

        failures.append({
            "failure_id": failure_id,
            "problem_id": record["problem_id"],
            "dataset": record["dataset"],
            "partition": record["partition"],
            "source_split": record["source_split"],
            "prompt": record["prompt"],
            "target": record["target"],
            "source_response": original,
            "corrupted_response": corrupted,
            "failure_kind": candidate["failure_kind"],
            "topology": candidate["topology"],
            "changed_spans": spans,
            "source_verifier": source_vr,
            "corrupted_verifier": corrupted_vr,
            "metadata": candidate["metadata"],
        })

print("Controlled failures retained:", len(failures))
print("Mutation attempts discarded:", len(discarded))

assert failures

## 14. Leakage and integrity audit

In [ ]:
source_df = pd.DataFrame(sources)
failure_df = pd.DataFrame(failures)
discarded_df = pd.DataFrame(discarded)

assert source_df["problem_id"].is_unique
assert failure_df["failure_id"].is_unique
assert set(failure_df["problem_id"]).issubset(
    set(source_df["problem_id"])
)

# All failures derived from one source remain in one partition.
assert int(
    failure_df.groupby("problem_id")["partition"].nunique().max()
) == 1

# Original public test examples remain confirmatory-only.
public_test = source_df[source_df["source_split"] == "test"]
assert set(public_test["partition"].unique()) == {"test"}

assert all(
    x["passed"] for x in failure_df["source_verifier"]
)
assert not any(
    x["passed"] for x in failure_df["corrupted_verifier"]
)

print("Leakage + integrity audit: PASS")

## 15. Summary tables

In [ ]:
print("SOURCE COUNTS")
source_summary = (
    source_df.groupby(["dataset", "partition"])
    .size()
    .rename("n")
    .reset_index()
)
display(source_summary)

print("\nFAILURE COUNTS")
failure_summary = (
    failure_df.groupby(
        ["dataset", "partition", "failure_kind", "topology"]
    )
    .size()
    .rename("n")
    .reset_index()
    .sort_values(["dataset", "partition", "failure_kind"])
)
display(failure_summary)

if len(discarded_df):
    print("\nDISCARDED MUTATION ATTEMPTS")
    discarded_summary = (
        discarded_df.groupby(["builder", "reason"])
        .size()
        .rename("n")
        .reset_index()
        .sort_values("n", ascending=False)
    )
    display(discarded_summary)

## 16. Human-readable spot audit

In [ ]:
rng = np.random.default_rng(SEED + 999)
n_show = min(6, len(failure_df))

for idx in rng.choice(
    len(failure_df), size=n_show, replace=False
):
    row = failure_df.iloc[int(idx)]

    print("=" * 100)
    print("ID:", row["failure_id"])
    print("Dataset:", row["dataset"])
    print("Partition:", row["partition"])
    print("Kind:", row["failure_kind"])
    print("\nPROMPT\n", row["prompt"])
    print("\nSOURCE\n", row["source_response"])
    print("\nCORRUPTED\n", row["corrupted_response"])
    print("\nCHANGED SPANS\n", row["changed_spans"])
    print()

## 17. Save immutable artifacts and manifest

In [ ]:
def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(
                json.dumps(
                    row, ensure_ascii=False, default=str
                )
                + "\n"
            )

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

sources_path = OUT_DIR / "repairbench_x_pilot_sources.jsonl"
failures_path = OUT_DIR / "repairbench_x_controlled_failures.jsonl"
discarded_path = OUT_DIR / "repairbench_x_discarded_mutations.jsonl"
summary_path = OUT_DIR / "repairbench_x_failure_summary.csv"

write_jsonl(sources_path, sources)
write_jsonl(failures_path, failures)
write_jsonl(discarded_path, discarded)
failure_summary.to_csv(summary_path, index=False)

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "03_RepairBench_X_Controlled_Failures_STANDALONE_FINAL",
    "seed": SEED,
    "dataset_revisions": {
        "openai/gsm8k": DATASET_SHAS["gsm8k"],
        "ChilleD/SVAMP": DATASET_SHAS["svamp"],
        "countdown_fresh": {
            "train_seed": SEED + 101,
            "calibration_seed": SEED + 102,
            "test_seed": SEED + 103,
        },
    },
    "counts": {
        "sources": len(sources),
        "controlled_failures": len(failures),
        "discarded_mutation_attempts": len(discarded),
    },
    "files": {
        sources_path.name: sha256_file(sources_path),
        failures_path.name: sha256_file(failures_path),
        discarded_path.name: sha256_file(discarded_path),
        summary_path.name: sha256_file(summary_path),
    },
    "primary_verifier": {
        "type": "strict_numeric_final_line",
        "required_format": "FINAL_ANSWER: <number>",
    },
    "math500_status": (
        "Deferred until symbolic-equivalence verification is validated."
    ),
}

manifest_path = OUT_DIR / "repairbench_x_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Saved:")
for p in [
    sources_path,
    failures_path,
    discarded_path,
    summary_path,
    manifest_path,
]:
    print(" -", p)

## 18. Final benchmark-ready gate

In [ ]:
required_failure_kinds = {
    "F1_final_numeric",
    "F3_verifier_marker",
    "F4_global_truncation",
}
observed_kinds = set(failure_df["failure_kind"])

per_dataset_test = (
    source_df[source_df["partition"] == "test"]
    .groupby("dataset")
    .size()
    .to_dict()
)

gate = {
    "source_ids_unique": bool(source_df["problem_id"].is_unique),
    "failure_ids_unique": bool(failure_df["failure_id"].is_unique),
    "all_sources_verified_correct": bool(
        all(x["passed"] for x in failure_df["source_verifier"])
    ),
    "all_failures_verified_wrong": bool(
        not any(x["passed"] for x in failure_df["corrupted_verifier"])
    ),
    "no_partition_leakage": bool(
        failure_df.groupby("problem_id")["partition"].nunique().max() == 1
    ),
    "minimum_failure_classes_present": bool(
        required_failure_kinds.issubset(observed_kinds)
    ),
    "test_sources_for_every_dataset": bool(
        all(
            per_dataset_test.get(ds, 0) > 0
            for ds in ["gsm8k", "svamp", "countdown_fresh"]
        )
    ),
}

gate["BENCHMARK_READY"] = all(gate.values())
gate["controlled_failure_count"] = int(len(failure_df))

gate_path = OUT_DIR / "BENCHMARK_READY.json"
gate_path.write_text(
    json.dumps(gate, indent=2),
    encoding="utf-8"
)

print(json.dumps(gate, indent=2))

if not gate["BENCHMARK_READY"]:
    raise RuntimeError(
        "RepairBench-X benchmark gate failed. "
        "Do not proceed to natural model failure collection."
    )

print("\nREPAIRBENCH-X CONTROLLED PILOT: PASS")

# Expected final result

A successful run ends with:

`REPAIRBENCH-X CONTROLLED PILOT: PASS`

The output directory will contain:
- `repairbench_x_pilot_sources.jsonl`
- `repairbench_x_controlled_failures.jsonl`
- `repairbench_x_discarded_mutations.jsonl`
- `repairbench_x_failure_summary.csv`
- `repairbench_x_manifest.json`
- `BENCHMARK_READY.json`

Notebook 04 can then collect **real natural failures** from one model at a time without rebuilding or changing this benchmark.